In [ ]:
from typing import Union
import torch
import os
from src.detector.detector_utils import huggingface_config, from_pretrained

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MessageList = list[dict[str, str]]
PromptInput = Union[str, MessageList, list[str], list[MessageList]]

cache_dir = os.getenv('CACHE_DIR')

config = {
    "params": {
        "model_name": "meta-llama/Llama-2-7b-hf",
        "dtype": torch.float16,
        "device": "cuda:1",
        "trust_remote_code": False,
        "padding_side": "left",
        "attn_implementation": "sdpa",
        "quantization_config": {
            "load_in_4bit": True,
            "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_compute_dtype": torch.float16,
            "bnb_4bit_use_duble_quant": True
        },
        "generation_params": {
            "max_new_tokens": 1024,
            "max_input_length": 4096,
            "do_sample": True,
            "temperature": 0.7,
            "top_p": 0.9,
            "repetition_penalty": 1.05,
        },
    }
}

In [ ]:
from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

import torch
import os
from src.detector.detector_utils import huggingface_config, from_pretrained


cache_dir = os.getenv('CACHE_DIR')
MODEL_ID = "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor_kwargs = {"token": huggingface_config["TOKEN"]}

model_kwargs = {
    "quantization_config": bnb_config,         # remplace dtype="auto"
    "device_map": {"": "cuda:1"},
    "attn_implementation": "sdpa",             # ou "flash_attention_2"
    "token": huggingface_config["TOKEN"],
}

processor = from_pretrained(cache_dir, AutoProcessor, MODEL_ID, processor_kwargs)
model = from_pretrained(cache_dir, AutoModelForCausalLM, MODEL_ID, model_kwargs)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explique l'entropie de Shannon."},
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        #cache_implementation="quantized",
        #cache_config={"backend": "quanto", "nbits": 4},
    )

response = processor.decode(out[0][input_len:], skip_special_tokens=False)
parsed = processor.parse_response(response)
print(parsed)